注意：运行下面的代码首先要执行 [gen_demo_factor_data.py](../gen_demo_factor_data.py) 脚本生成示例数据。

In [33]:
import warnings
warnings.filterwarnings('ignore')
import logging
import datetime as dt

import numpy as np
import pandas as pd

from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)
from QuantStudio.Tools.Visualization import qs_help
from QuantStudio.Factor.HDF5DB import HDF5DB

FDB = HDF5DB(args={"MainDir": "../data/HDF5"}).connect()

# 因子定义框架

因子定义是基于[计算图框架](../基本框架.ipynb#ComputationGraph)构建的，每个因子是一个计算节点。因子的定义逻辑上构成一个有向无环的计算图(DAG)。

所有的因子划分成两类:
* **基础因子**: 指直接由原始数据转化而来, 并不依赖于其他因子的因子, 比如最新股东权益因子就是以财务报表里的股东权益合计这个数据项转化而来的基础因子, 大多数的基础因子通过因子表的 `getFactor` 方法或者直接由数据通过 `DataFactor`（参见 [数据因子](#数据因子)）构造得到
* **衍生因子**: 依赖于其他因子通过因子运算得到的因子, 比如市盈率因子则是用总市值因子除以最新股东权益因子运算得来的衍生因子.

因子的运算分成四类: **单点运算**, **时序运算**, **截面运算**以及**面板运算**. 每种运算的适用范围各不相同, 在效率上也有所差异. 这些运算可以嵌套组合以形成更复杂的计算.

以 HSIGMA 因子的定义举例说明. HSIGMA 因子是 Barra 中国市场风险模型(CNE5)里的一个风险因子, 称为历史残余波动率因子. 该因子的构造过程如下：
* 首先由股票收盘价和昨收盘价因子计算收益率因子，这是单点运算;
* 然后用股票的收益率因子和市场收益率因子进行时间序列回归得到残余收益率因子 EPSILON, 这是时序运算;
* 最后取一段时间的残余收益率求其标准差得到残余波动率因子 HSIGMA，这也是时序运算. 

具体定义如下图所示:

![因子定义模型](../images/因子定义模型.png)

# 因子运算

## 因子算子

构建衍生因子最直接的方法是将因子算子对象作用在相应的因子对象上，因子算子均继承自 `QuantStudio.Factor.FactorOperation.FactorOperator`，FactorOperator 有四个基础的子类分别代表不同的运算类型：
* `QuantStudio.Factor.FactorOperation.PointOperator`: 单点运算算子
* `QuantStudio.Factor.FactorOperation.TimeOperator`: 时序运算算子
* `QuantStudio.Factor.FactorOperation.SectionOperator`: 截面运算算子
* `QuantStudio.Factor.FactorOperation.PanelOperator`: 面板运算算子

每个具体的算子必须实现类方法 `calculate` 来实现具体的运算逻辑

In [19]:
from QuantStudio.Factor.FactorOperation import FactorOperator

print(qs_help(FactorOperator.calculate))

类型: function
模块: QuantStudio.Factor.FactorOperation
签名: FactorOperator.calculate(self, f: QuantStudio.Factor.Factor.Factor, idt: Union[datetime.datetime, List[datetime.datetime]], iid: Union[str, List[str]], x: list, args: dict)
说明文档:
    算子的运算逻辑实现
    
    Args:
        f: 该算子所属的因子对象
        idt: 当前待计算的时点
        iid: 当前待计算的 ID
        x: 描述子当期的数据
        args: 计算需要的附加参数, 来自于算子和因子对象的 ModelArgs, {参数名: 参数值}
    
    Returns:
        在时点 idt, ID 为 iid 的因子值


创建自定义算子有两个途径：
* 根据不同的运算类型，创建对应于 `PointOperator, TimeOperator, SectionOperator, PanelOperator` 的子类，实现其 `calculdate` 方法，这种方式比较麻烦，尽量避免使用。
* 通过一个函数直接创建算子，可以使用 `makeFactorOperator` 工厂函数或者 `FactorOperatorized` 装饰器来将给定的函数转换为因子算子。给定的函数将作为算子对象的 `calculate` 实现。这是最常用的方式。

In [ ]:
# makeFactorOperator 算子工厂函数
from QuantStudio.Factor.FactorOperation import makeFactorOperator

print(qs_help(makeFactorOperator))

类型: function
模块: QuantStudio.Factor.FactorOperation
签名: makeFactorOperator(func: Callable[[QuantStudio.Factor.Factor.Factor, Union[datetime.datetime, List[datetime.datetime]], Union[str, List[str]], list, dict], Any], operator_type: Literal['Point', 'Time', 'Section', 'Panel'], args: dict = {}, **kwargs) -> QuantStudio.Factor.FactorOperation.FactorOperator
说明文档:
    算子工厂函数, 给定一个函数创建一个因子算子对象
    
    Args:
        func: 定义了算子运算逻辑的函数
        operator_type: 算子类型
        args: 创建算子对象时传递给它的参数集
        kwargs: 创建算子对象时传递给它的其他入参
    
    Returns:
        创建的因子算子对象


In [26]:
# makeFactorOperator 工厂函数创建算子
from QuantStudio.Factor.FactorOperation import makeFactorOperator

def calcMid(f, idt, iid, x, args):
    """计算两个因子的中间值"""
    return (x[0] + x[1]) / 2

calcMid = makeFactorOperator(func=calcMid, operator_type="Point", args={"Name": "calcMid", "Arity": 2})
print(qs_help(calcMid))

类型: PointOperator
模块: QuantStudio.Factor.FactorOperation
QS 对象类型: 因子算子
QS 对象名称: calcMid
QSID: e26c32f3c873cea6748b54ad47cc294c1a92304ecc02997b3928d7605cf78272
参数集:
    * OperatorType(算子类型): typing.Literal['Point'], 默认值 'Point', 当前取值: 'Point'
    * Name(名称): <class 'str'>, 默认值 'PointOperator', 当前取值: 'calcMid'
    * ModelArgs(模型参数): typing.Dict[str, typing.Any], 默认值 {}, 当前取值: {}
    * Arity(入参数): typing.Optional[int], 默认值 None, 当前取值: 2
    * DataType(数据类型): typing.Literal['double', 'string', 'object'], 默认值 'double', 当前取值: 'double'
    * Description(描述信息): <class 'str'>, 默认值 '', 当前取值: ''
    * Meta(元信息): typing.Dict[str, typing.Any], 默认值 {}, 当前取值: {}
    * InputFormat(输入格式): typing.Literal['numpy', 'pandas'], 默认值 'numpy', 当前取值: 'numpy'
    * ExpandDescriptors(展开描述子): typing.List[int], 默认值 [], 当前取值: []
    * DescriptorCompoundType(描述子复合类型): typing.List[typing.List[typing.Tuple[str, typing.Literal['double', 'string', 'object']]]], 默认值 [], 当前取值: []
    * MultiMapping(多重映射): <class 'bool'>, 默

In [22]:
# FactorOperatorized 装饰器
from QuantStudio.Factor.FactorOperation import FactorOperatorized

print(qs_help(FactorOperatorized))

类型: function
模块: QuantStudio.Factor.FactorOperation
签名: FactorOperatorized(operator_type: Literal['Point', 'Time', 'Section', 'Panel'], args: dict = {}, **kwargs) -> Callable[[Callable[[QuantStudio.Factor.Factor.Factor, Union[datetime.datetime, List[datetime.datetime]], Union[str, List[str]], list, dict], Any]], QuantStudio.Factor.FactorOperation.FactorOperator]
说明文档:
    将函数转换成因子算子对象的装饰器
    
    Args:
        operator_type: 算子类型
        args: 创建算子对象时传递给它的参数集
        kwargs: 创建算子对象时传递给它的其他入参
    
    Returns:
        算子装饰器


In [27]:
# FactorOperatorized 装饰器创建算子
from QuantStudio.Factor.FactorOperation import FactorOperatorized

@FactorOperatorized(operator_type="Point", args={"Name": "calcMid", "Arity": 2})
def calcMid(f, idt, iid, x, args):
    """计算两个因子的中间值"""
    return (x[0] + x[1]) / 2

print(qs_help(calcMid))

类型: PointOperator
模块: QuantStudio.Factor.FactorOperation
QS 对象类型: 因子算子
QS 对象名称: calcMid
QSID: e26c32f3c873cea6748b54ad47cc294c1a92304ecc02997b3928d7605cf78272
参数集:
    * OperatorType(算子类型): typing.Literal['Point'], 默认值 'Point', 当前取值: 'Point'
    * Name(名称): <class 'str'>, 默认值 'PointOperator', 当前取值: 'calcMid'
    * ModelArgs(模型参数): typing.Dict[str, typing.Any], 默认值 {}, 当前取值: {}
    * Arity(入参数): typing.Optional[int], 默认值 None, 当前取值: 2
    * DataType(数据类型): typing.Literal['double', 'string', 'object'], 默认值 'double', 当前取值: 'double'
    * Description(描述信息): <class 'str'>, 默认值 '', 当前取值: ''
    * Meta(元信息): typing.Dict[str, typing.Any], 默认值 {}, 当前取值: {}
    * InputFormat(输入格式): typing.Literal['numpy', 'pandas'], 默认值 'numpy', 当前取值: 'numpy'
    * ExpandDescriptors(展开描述子): typing.List[int], 默认值 [], 当前取值: []
    * DescriptorCompoundType(描述子复合类型): typing.List[typing.List[typing.Tuple[str, typing.Literal['double', 'string', 'object']]]], 默认值 [], 当前取值: []
    * MultiMapping(多重映射): <class 'bool'>, 默

因子算子实现了 `__call__` 方法，即是 Callable 对象，将其作用在对应的因子上即可创建新的衍生因子。

另外，衍生因子具有属性 `Operator`, 即是构造它的算子对象。

In [31]:
# 通过算子创建衍生因子
FT = FDB.getTable("stock_cn_day_bar")
High, Low = FT.getFactor("high"), FT.getFactor("low")

Mid = calcMid(High, Low, factor_args={"Name": "Mid"})
print(qs_help(Mid))

类型: PointOperation
模块: QuantStudio.Factor.FactorOperation
QS 对象类型: 计算节点-因子
QS 对象名称: Mid
QSID: 41e6b74f74f8952533ed83597365690f36a413f3c8e34c0c3b3b20a52665096a
参数集:
    * Name(名称): <class 'str'>, 默认值 'PointOperation', 当前取值: 'Mid'
    * Meta(元信息): <class 'dict'>, 默认值 {}, 当前取值: {}
    * SectionIDs(截面ID): typing.Optional[typing.List[str]], 默认值 None, 当前取值: None
    * CalcDTRuler(计算时点标尺): typing.Optional[typing.List[datetime.datetime]], 默认值 None, 当前取值: None
    * CacheEnabled(启用缓存): <class 'bool'>, 默认值 True, 当前取值: True
    * Operator(算子): <class 'QuantStudio.Factor.FactorOperation.PointOperator'>, 无默认值, 当前取值: <QuantStudio.Factor.FactorOperation.PointOperator object at 0x7c50cdb5aba0>
    * ModelArgs(参数): <class 'dict'>, 默认值 {}, 当前取值: {}
说明文档:
    基于单点算子的衍生因子


## 单点运算

固定时点、固定 ID 的不同因子间的运算. 单点运算是最简单的一类因子运算, 其典型例子是各种估值因子的定义, 比如市净率(BP)因子, 其是由股东权益除以总市值得到. BP 因子的描述子即为股东权益因子和总市值因子, 而定义算子是一个除法运算(真正的实现可能还要包括缺失值处理以及对于分母为 0 的处理).

In [37]:
from QuantStudio.Factor.FactorOperation import PointOperator
print(qs_help(PointOperator.calculate))

类型: function
模块: QuantStudio.Factor.FactorOperation
签名: PointOperator.calculate(self, f: QuantStudio.Factor.Factor.Factor, idt: Union[datetime.datetime, List[datetime.datetime]], iid: Union[str, List[str]], x: list, args: dict)
说明文档:
    算子的运算逻辑实现
    
    Args:
        f: 该算子所属的因子对象
        idt: 当前待计算的时点, 如果 DTMode 为多时点, 则该值为时点序列 list[datetime]
        iid: 当前待计算的 ID, 如果 IDMode 为多ID, 则该值为 ID 序列 list[str], 注意并发时 iid 并不一定是全截面
        x: 描述子当期的数据, [单个描述子值 or array]
            * 如果 DTMode 为单时点, IDMode 为单ID, 那么 x 元素为单个描述子值, 同时方法需返回单个元素
            * 如果 DTMode 为单时点, IDMode 为多ID, 那么 x 元素为 array(shape=(len(iid), )), 同时方法需返回 array(shape=(len(iid), ))
            * 如果 DTMode 为多时点, IDMode 为单ID, 那么 x 元素为 array(shape=(len(idt), )), 同时方法需返回 array(shape=(len(idt), ))
            * 如果 DTMode 为多时点, IDMode 为多ID, 那么 x 元素为 array(shape=(len(idt), len(iid))), 同时方法需返回 array(shape=(len(idt), len(iid)))
        args: 计算需要附加的模型参数, 来自于算子和因子对象的 ModelArgs, {参数名: 参数值}
    
    Returns:
        在时点 idt, ID 为 iid 的

单点算子的主要参数是 DTMode, IDMode, 其不同的取值会影响传入计算函数的入参，下面是对于这两个参数不同的组合后计算函数的入参变化。

In [38]:
# 单点运算, 参数 DTMode="多时点", IDMode="多ID" 模式下
FT = FDB.getTable(table_name="stock_cn_day_bar")
High, Low = FT.getFactor("high"), FT.getFactor("low")

@FactorOperatorized(operator_type="Point", args={"Name": "calcMid", "Arity": 2, "DataType": "double", "DTMode": "多时点", "IDMode": "多ID", "ModelArgs": {}})
def PointFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")
    for i, ix in enumerate(x):
        print(f"x[{i}] : {ix}")
    print(f"args : {args}")
    print("-" * 10)
    return (x[0] + x[1]) / 2

PointFactor = PointFun(High, Low)

IDs = ["000001.SZ", "000002.SZ"]
DTs = FT.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 3))
print(PointFactor.readData(ids=IDs, dts=DTs))

idt : [datetime.datetime(2025, 1, 1, 0, 0), datetime.datetime(2025, 1, 2, 0, 0), datetime.datetime(2025, 1, 3, 0, 0)]
iid : ['000001.SZ', '000002.SZ']
x[0] : [[8.11518471 7.15189366]
 [9.78618342 7.99158564]
 [6.01943698 4.37031954]]
x[1] : [[5.48813504 1.80202707]
 [0.35219805 1.80660622]
 [3.59507901 0.63369005]]
args : {}
----------
            000001.SZ  000002.SZ
2025-01-01   6.801660   4.476960
2025-01-02   5.069191   4.899096
2025-01-03   4.807258   2.502005


In [39]:
# 单点运算, 参数 DTMode="单时点", IDMode="多ID" 模式下
FT = FDB.getTable(table_name="stock_cn_day_bar")
High, Low = FT.getFactor("high"), FT.getFactor("low")

@FactorOperatorized(operator_type="Point", args={"Name": "calcMid", "Arity": 2, "DataType": "double", "DTMode": "单时点", "IDMode": "多ID", "ModelArgs": {}})
def PointFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")
    for i, ix in enumerate(x):
        print(f"x[{i}] : {ix}")
    print(f"args : {args}")
    print("-" * 10)
    return (x[0] + x[1]) / 2

PointFactor = PointFun(High, Low)

IDs = ["000001.SZ", "000002.SZ"]
DTs = FT.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 3))
print(PointFactor.readData(ids=IDs, dts=DTs))

idt : 2025-01-01 00:00:00
iid : ['000001.SZ', '000002.SZ']
x[0] : [8.11518471 7.15189366]
x[1] : [5.48813504 1.80202707]
args : {}
----------
idt : 2025-01-02 00:00:00
iid : ['000001.SZ', '000002.SZ']
x[0] : [9.78618342 7.99158564]
x[1] : [0.35219805 1.80660622]
args : {}
----------
idt : 2025-01-03 00:00:00
iid : ['000001.SZ', '000002.SZ']
x[0] : [6.01943698 4.37031954]
x[1] : [3.59507901 0.63369005]
args : {}
----------
            000001.SZ  000002.SZ
2025-01-01   6.801660   4.476960
2025-01-02   5.069191   4.899096
2025-01-03   4.807258   2.502005


In [40]:
# 单点运算, 参数 DTMode="多时点", IDMode="单ID" 模式下
FT = FDB.getTable(table_name="stock_cn_day_bar")
High, Low = FT.getFactor("high"), FT.getFactor("low")

@FactorOperatorized(operator_type="Point", args={"Name": "calcMid", "Arity": 2, "DataType": "double", "DTMode": "多时点", "IDMode": "单ID", "ModelArgs": {}})
def PointFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")
    for i, ix in enumerate(x):
        print(f"x[{i}] : {ix}")
    print(f"args : {args}")
    print("-" * 10)
    return (x[0] + x[1]) / 2

PointFactor = PointFun(High, Low)

IDs = ["000001.SZ", "000002.SZ"]
DTs = FT.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 3))
print(PointFactor.readData(ids=IDs, dts=DTs))

idt : [datetime.datetime(2025, 1, 1, 0, 0), datetime.datetime(2025, 1, 2, 0, 0), datetime.datetime(2025, 1, 3, 0, 0)]
iid : 000001.SZ
x[0] : [8.11518471 9.78618342 6.01943698]
x[1] : [5.48813504 0.35219805 3.59507901]
args : {}
----------
idt : [datetime.datetime(2025, 1, 1, 0, 0), datetime.datetime(2025, 1, 2, 0, 0), datetime.datetime(2025, 1, 3, 0, 0)]
iid : 000002.SZ
x[0] : [7.15189366 7.99158564 4.37031954]
x[1] : [1.80202707 1.80660622 0.63369005]
args : {}
----------
            000001.SZ  000002.SZ
2025-01-01   6.801660   4.476960
2025-01-02   5.069191   4.899096
2025-01-03   4.807258   2.502005


In [41]:
# 单点运算, 参数 DTMode="单时点", IDMode="单ID" 模式下
FT = FDB.getTable(table_name="stock_cn_day_bar")
High, Low = FT.getFactor("high"), FT.getFactor("low")

@FactorOperatorized(operator_type="Point", args={"Name": "calcMid", "Arity": 2, "DataType": "double", "DTMode": "单时点", "IDMode": "单ID", "ModelArgs": {}})
def PointFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")
    for i, ix in enumerate(x):
        print(f"x[{i}] : {ix}")
    print(f"args : {args}")
    print("-" * 10)
    return (x[0] + x[1]) / 2

PointFactor = PointFun(High, Low)

IDs = ["000001.SZ", "000002.SZ"]
DTs = FT.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 3))
print(PointFactor.readData(ids=IDs, dts=DTs))

idt : 2025-01-01 00:00:00
iid : 000001.SZ
x[0] : 8.115184706218832
x[1] : 5.4881350392732475
args : {}
----------
idt : 2025-01-01 00:00:00
iid : 000002.SZ
x[0] : 7.151893663724195
x[1] : 1.8020270731629007
args : {}
----------
idt : 2025-01-02 00:00:00
iid : 000001.SZ
x[0] : 9.78618342232764
x[1] : 0.35219804575407077
args : {}
----------
idt : 2025-01-02 00:00:00
iid : 000002.SZ
x[0] : 7.9915856421672355
x[1] : 1.8066062154063434
args : {}
----------
idt : 2025-01-03 00:00:00
iid : 000001.SZ
x[0] : 6.019436981131124
x[1] : 3.59507900573786
args : {}
----------
idt : 2025-01-03 00:00:00
iid : 000002.SZ
x[0] : 4.3703195379934145
x[1] : 0.6336900454884831
args : {}
----------
            000001.SZ  000002.SZ
2025-01-01   6.801660   4.476960
2025-01-02   5.069191   4.899096
2025-01-03   4.807258   2.502005


## 时序运算

固定 ID 不同因子间在时间序列上的运算. 除了指定描述子和定义算子外, 时间序列运算还需要指定每个描述子的回溯期数. 时间序列运算的典型例子是移动平均线的定义, 其是由证券过去一段时间的价格序列取某种形式的平均得到的. 移动平均线因子的描述子即为价格因子, 而定义算子是一个平均值运算.

In [49]:
from QuantStudio.Factor.FactorOperation import TimeOperator
print(qs_help(TimeOperator.calculate))

类型: function
模块: QuantStudio.Factor.FactorOperation
签名: TimeOperator.calculate(self, f: QuantStudio.Factor.Factor.Factor, idt: Union[datetime.datetime, List[datetime.datetime]], iid: Union[str, List[str]], x: list, args: dict)
说明文档:
    算子的运算逻辑实现
    
    Args:
        f: 该算子所属的因子对象
        idt: 当前待计算的时点, 如果 DTMode 为多时点，则该值为时点序列 list[datetime]
        iid: 当前待计算的 ID, 如果 IDMode 为多ID, 则该值为 ID 序列 list[str], 注意并发时 iid 并不一定是全截面
        x: 描述子当期的数据, [array]
            * 如果 DTMode 为单时点, IDMode 为单ID, 那么 x 的第 i 个元素为 array(shape=(LookBack[i]+1, )), 同时方法需返回单个元素
            * 如果 DTMode 为单时点, IDMode 为多ID, 那么 x 的第 i 个元素为 array(shape=(LookBack[i]+1, len(iid))), 同时方法需返回 array(shape=(len(iid), ))
            * 如果 DTMode 为多时点, IDMode 为单ID, 那么 x 的第 i 个元素为 array(shape=(LookBack[i]+len(idt), )), 同时方法需返回返回 array(shape=(nDate,))
            * 如果 DTMode 为多时点, IDMode 为多ID, 那么 x 的第 i 个元素为 array(shape=(LookBack[i]+len(idt), len(iid))), 同时方法需返回 array(shape=(len(idt), len(iid)))
        args: 计算需要附加的模型参数, 来自于算子和因

时序算子的主要参数是 DTMode, IDMode, LookBack, StartDT, iInitFactor, 其不同的取值会影响传入计算函数的入参，其中 DTMode, IDMode 跟单点算子的作用类似。下面是对于后面三个参数不同的组合后计算函数的入参变化。

In [54]:
# 时间序列运算: 无自身迭代, 滚动窗口模式 (iInitFactor<0, StartDT 均为 None)
from QuantStudio.Factor.FactorOperation import FactorOperatorized

FT = FDB.getTable(table_name="stock_cn_day_bar")
Close = FT.getFactor("close")

@FactorOperatorized(operator_type="Time", args={"Name": "calcMA", "Arity": 1, "DataType": "double", "LookBack": [3-1], "StartDT": [None], "iInitFactor": -1, "DTMode": "多时点", "IDMode": "多ID", "ModelArgs": {}})
def TimeFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")
    for i, ix in enumerate(x):
        print(f"x[{i}] : {ix}")
    print(f"args : {args}")
    print("-" * 10)
    w = f.Operator.Args["LookBack"][0]
    return pd.DataFrame(x[0]).rolling(window=w+1).mean().values[w:]

TimeFactor = TimeFun(Close, factor_args={"Name": "MA"})

IDs = ["000001.SZ", "000002.SZ"]
DTRuler = FT.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 10))
DTs = DTRuler[-3:]
print(TimeFactor.readData(ids=IDs, dts=DTs, dt_ruler=DTRuler))

idt : [datetime.datetime(2025, 1, 6, 0, 0), datetime.datetime(2025, 1, 7, 0, 0), datetime.datetime(2025, 1, 8, 0, 0), datetime.datetime(2025, 1, 9, 0, 0), datetime.datetime(2025, 1, 10, 0, 0)]
iid : ['000001.SZ', '000002.SZ']
x[0] : [[2.30400314 4.26691412]
 [7.08360267 3.92758946]
 [8.16825141 5.39536605]
 [3.18402935 7.28947696]
 [9.15652841 0.19232059]]
args : {}
----------
            000001.SZ  000002.SZ
2025-01-08   5.851952   4.529957
2025-01-09   6.145294   5.537477
2025-01-10   6.836270   4.292388


In [59]:
# 时间序列运算: 无自身迭代, 扩张窗口模式(iInitFactor<0, StartDT 不为 None)
from QuantStudio.Factor.FactorOperation import FactorOperatorized

FT = FDB.getTable(table_name="stock_cn_day_bar")
Close = FT.getFactor("close")

@FactorOperatorized(operator_type="Time", args={"Name": "calcMA", "Arity": 1, "DataType": "double", "LookBack": [1-1], "StartDT": [dt.datetime(2025, 1, 5)], "iInitFactor": -1, "DTMode": "单时点", "IDMode": "多ID", "ModelArgs": {}})
def TimeFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")
    for i, ix in enumerate(x):
        print(f"x[{i}] : {ix}")
    print(f"args : {args}")
    print("-" * 10)
    w = f.Operator.Args["LookBack"][0]
    return pd.DataFrame(x[0]).mean().values[w:]

TimeFactor = TimeFun(Close, factor_args={"Name": "MA"})

IDs = ["000001.SZ", "000002.SZ"]
DTRuler = FT.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 10))
DTs = DTRuler[-3:]
print(TimeFactor.readData(ids=IDs, dts=DTs, dt_ruler=DTRuler))

idt : [datetime.datetime(2025, 1, 5, 0, 0), datetime.datetime(2025, 1, 6, 0, 0), datetime.datetime(2025, 1, 7, 0, 0), datetime.datetime(2025, 1, 8, 0, 0)]
iid : ['000001.SZ', '000002.SZ']
x[0] : [[4.88442467 1.34267024]
 [2.30400314 4.26691412]
 [7.08360267 3.92758946]
 [8.16825141 5.39536605]]
args : {}
----------
idt : [datetime.datetime(2025, 1, 5, 0, 0), datetime.datetime(2025, 1, 6, 0, 0), datetime.datetime(2025, 1, 7, 0, 0), datetime.datetime(2025, 1, 8, 0, 0), datetime.datetime(2025, 1, 9, 0, 0)]
iid : ['000001.SZ', '000002.SZ']
x[0] : [[4.88442467 1.34267024]
 [2.30400314 4.26691412]
 [7.08360267 3.92758946]
 [8.16825141 5.39536605]
 [3.18402935 7.28947696]]
args : {}
----------
idt : [datetime.datetime(2025, 1, 5, 0, 0), datetime.datetime(2025, 1, 6, 0, 0), datetime.datetime(2025, 1, 7, 0, 0), datetime.datetime(2025, 1, 8, 0, 0), datetime.datetime(2025, 1, 9, 0, 0), datetime.datetime(2025, 1, 10, 0, 0)]
iid : ['000001.SZ', '000002.SZ']
x[0] : [[4.88442467 1.34267024]
 [2.30400

In [ ]:
# 时间序列运算: 自身迭代, 滚动窗口模式 (iInitFactor>=0, StartDT 均为 None)
from QuantStudio.Factor.FactorOperation import FactorOperatorized

FT = FDB.getTable(table_name="stock_cn_day_bar")
Close = FT.getFactor("close")

@FactorOperatorized(operator_type="Time", args={"Name": "calcEMA", "Arity": 2, "DataType": "double", "LookBack": [2-1, 1-1], "StartDT": [None, None], "iInitFactor": 0, "DTMode": "单时点", "IDMode": "多ID", "ModelArgs": {}})
def TimeFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")
    for i, ix in enumerate(x):
        print(f"x[{i}] : {ix}")
    print(f"args : {args}")
    print("-" * 10)
    return 0.5 * x[0][0] + 0.5 * x[1][0]

TimeFactor = TimeFun(1, Close, factor_args={"Name": "EMA"})

IDs = ["000001.SZ", "000002.SZ"]
DTRuler = FT.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 10))
DTs = DTRuler[-3:]
print(TimeFactor.readData(ids=IDs, dts=DTs, dt_ruler=DTRuler))

2026-03-23 14:48:42,055 | QS | WARNING : 算子 calcEMA(QSID: 94133fdf88f72959112b0e53c45c4ecb7c5ef3958bb338ca54bdf2e519177275) 为自身迭代且滚动窗口模式，在缓存的不同状态下产生的数据会不一致，所以该算子作用的因子将强制不使用缓存!
2026-03-23 14:48:42,055 | QS | WARNING : 算子 calcEMA(QSID: 94133fdf88f72959112b0e53c45c4ecb7c5ef3958bb338ca54bdf2e519177275) 为自身迭代且滚动窗口模式，在缓存的不同状态下产生的数据会不一致，所以该算子作用的因子将强制不使用缓存!


idt : [datetime.datetime(2025, 1, 7, 0, 0), datetime.datetime(2025, 1, 8, 0, 0)]
iid : ['000001.SZ', '000002.SZ']
x[0] : [[1.0 1.0]
 [nan nan]]
x[1] : [[8.16825141 5.39536605]]
args : {}
----------
idt : [datetime.datetime(2025, 1, 8, 0, 0), datetime.datetime(2025, 1, 9, 0, 0)]
iid : ['000001.SZ', '000002.SZ']
x[0] : [[4.584125707392189 3.197683025548154]
 [nan nan]]
x[1] : [[3.18402935 7.28947696]]
args : {}
----------
idt : [datetime.datetime(2025, 1, 9, 0, 0), datetime.datetime(2025, 1, 10, 0, 0)]
iid : ['000001.SZ', '000002.SZ']
x[0] : [[3.884077526749032 5.243579993491191]
 [nan nan]]
x[1] : [[9.15652841 0.19232059]]
args : {}
----------
           000001.SZ 000002.SZ
2025-01-08  4.584126  3.197683
2025-01-09  3.884078   5.24358
2025-01-10  6.520303   2.71795


In [61]:
# 时间序列运算: 自身迭代, 扩张窗口模式 (iInitFactor>=0, StartDT 不为 None)
from QuantStudio.Factor.FactorOperation import FactorOperatorized

FT = FDB.getTable(table_name="stock_cn_day_bar")
Close = FT.getFactor("close")

@FactorOperatorized(operator_type="Time", args={"Name": "TimeFun", "Arity": 2, "DataType": "double", "LookBack": [2-1, 1-1], "StartDT": [dt.datetime(2025, 1, 5), dt.datetime(2025, 1, 5)], "iInitFactor": 0, "DTMode": "单时点", "IDMode": "多ID", "ModelArgs": {}})
def TimeFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")
    for i, ix in enumerate(x):
        print(f"x[{i}] : {ix}")
    print(f"args : {args}")
    print("-" * 10)
    return 0.5 * np.nanmean(x[0][:-1], axis=0) + 0.5 * np.nanmean(x[1], axis=0)

TimeFactor = TimeFun(1, Close)

IDs = ["000001.SZ", "000002.SZ"]
DTRuler = FT.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 10))
DTs = DTRuler[-3:]
print(TimeFactor.readData(ids=IDs, dts=DTs, dt_ruler=DTRuler))

idt : [datetime.datetime(2025, 1, 4, 0, 0), datetime.datetime(2025, 1, 5, 0, 0)]
iid : ['000001.SZ', '000002.SZ']
x[0] : [[1.0 1.0]
 [nan nan]]
x[1] : [[4.88442467 1.34267024]]
args : {}
----------
idt : [datetime.datetime(2025, 1, 4, 0, 0), datetime.datetime(2025, 1, 5, 0, 0), datetime.datetime(2025, 1, 6, 0, 0)]
iid : ['000001.SZ', '000002.SZ']
x[0] : [[1.0 1.0]
 [2.9422123336513795 1.1713351185891243]
 [nan nan]]
x[1] : [[4.88442467 1.34267024]
 [2.30400314 4.26691412]]
args : {}
----------
idt : [datetime.datetime(2025, 1, 4, 0, 0), datetime.datetime(2025, 1, 5, 0, 0), datetime.datetime(2025, 1, 6, 0, 0), datetime.datetime(2025, 1, 7, 0, 0)]
iid : ['000001.SZ', '000002.SZ']
x[0] : [[1.0 1.0]
 [2.9422123336513795 1.1713351185891243]
 [2.7826600360064737 1.9452298694016432]
 [nan nan]]
x[1] : [[4.88442467 1.34267024]
 [2.30400314 4.26691412]
 [7.08360267 3.92758946]]
args : {}
----------
idt : [datetime.datetime(2025, 1, 4, 0, 0), datetime.datetime(2025, 1, 5, 0, 0), datetime.datetim

## 截面运算

固定时点不同因子间在横截面上的运算. 横截面运算的典型例子是各种因子数据标准化的处理. 比如 Z-score 标准化方法, 即是用每只证券的因子值减去整个截面因子值的平均数并处以截面标准差所得. 

In [62]:
from QuantStudio.Factor.FactorOperation import SectionOperator
print(qs_help(SectionOperator.calculate))

类型: function
模块: QuantStudio.Factor.FactorOperation
签名: SectionOperator.calculate(self, f: QuantStudio.Factor.Factor.Factor, idt: Union[datetime.datetime, List[datetime.datetime]], iid: Union[str, List[str]], x: list, args: dict)
说明文档:
    算子的运算逻辑实现
    
    Args:
        f: 该算子所属的因子对象
        idt: 当前待计算的时点, 如果 DTMode 为多时点, 则该值为时点序列 list[datetime]
        iid: 当前待计算的 ID, 如果 OutputMode 为全截面, 则该值为 ID 序列 list[str], 该序列在并发时也是全体截面 ID
        x: 描述子当期的数据, [array]
            * 如果 DTMode 为单时点, 那么 x 元素为 array(shape=(len(iid), )), 如果输出形式为全截面返回 array(shape=(len(iid), )), 否则返回单个值
            * 如果 DTMode 为多时点, 那么 x 元素为 array(shape=(len(idt), len(iid))), 如果输出形式为全截面返回 array(shape=(len(idt), len(iid))), 否则返回 array(shape=(len(idt), ))
        args: 计算需要附加的模型参数, 来自于算子和因子对象的 ModelArgs, {参数名: 参数值}
    
    Returns:
        在时点 idt, ID 为 iid 的因子值


截面算子的主要参数是 DTMode, 其不同的取值会影响传入计算函数的入参，其中 DTMode 跟单点算子的作用类似。下面是对于这个参数不同的取值计算函数的入参变化。

In [68]:
# 截面运算: DTMode="单时点" 模式
from QuantStudio.Factor.FactorOperation import FactorOperatorized

FT = FDB.getTable(table_name="stock_cn_day_bar")
Close = FT.getFactor("close")

@FactorOperatorized(operator_type="Section", args={"Name": "SectionFun", "Arity": 1, "DataType": "double", "DTMode": "单时点", "ModelArgs": {}})
def SectionFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")
    for i, ix in enumerate(x):
        print(f"x[{i}] : {ix}")
    print(f"args : {args}")
    print("-" * 10)
    return x[0] - np.nanmean(x[0])

SectionFactor = SectionFun(Close)

SectionIDs = ["000001.SZ", "000002.SZ", "000003.SZ"]
IDs = ["000001.SZ", "000002.SZ"]
DTs = FT.getDateTime(start_dt=dt.datetime(2025, 1, 8), end_dt=dt.datetime(2025, 1, 10))
print(SectionFactor.readData(ids=IDs, dts=DTs, section_ids=SectionIDs))

idt : 2025-01-08 00:00:00
iid : ['000001.SZ', '000002.SZ', '000003.SZ']
x[0] : [8.16825141 5.39536605 0.45850355]
args : {}
----------
idt : 2025-01-09 00:00:00
iid : ['000001.SZ', '000002.SZ', '000003.SZ']
x[0] : [3.18402935 7.28947696 5.69195972]
args : {}
----------
idt : 2025-01-10 00:00:00
iid : ['000001.SZ', '000002.SZ', '000003.SZ']
x[0] : [9.15652841 0.19232059 5.69872151]
args : {}
----------
            000001.SZ  000002.SZ
2025-01-08   3.494211   0.721326
2025-01-09  -2.204459   1.900988
2025-01-10   4.140672  -4.823536


In [67]:
# 截面运算: DTMode="多时点" 模式
from QuantStudio.Factor.FactorOperation import FactorOperatorized

FT = FDB.getTable(table_name="stock_cn_day_bar")
Close = FT.getFactor("close")

@FactorOperatorized(operator_type="Section", args={"Name": "SectionFun", "Arity": 1, "DataType": "double", "DTMode": "多时点", "ModelArgs": {}})
def SectionFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")
    for i, ix in enumerate(x):
        print(f"x[{i}] : {ix}")
    print(f"args : {args}")
    print("-" * 10)
    return x[0] - np.nanmean(x[0], axis=1, keepdims=True)

SectionFactor = SectionFun(Close)

SectionIDs = ["000001.SZ", "000002.SZ", "000003.SZ"]
IDs = ["000001.SZ", "000002.SZ"]
DTs = FT.getDateTime(start_dt=dt.datetime(2025, 1, 8), end_dt=dt.datetime(2025, 1, 10))
print(SectionFactor.readData(ids=IDs, dts=DTs, section_ids=SectionIDs))

idt : [datetime.datetime(2025, 1, 8, 0, 0), datetime.datetime(2025, 1, 9, 0, 0), datetime.datetime(2025, 1, 10, 0, 0)]
iid : ['000001.SZ', '000002.SZ', '000003.SZ']
x[0] : [[8.16825141 5.39536605 0.45850355]
 [3.18402935 7.28947696 5.69195972]
 [9.15652841 0.19232059 5.69872151]]
args : {}
----------
            000001.SZ  000002.SZ
2025-01-08   3.494211   0.721326
2025-01-09  -2.204459   1.900988
2025-01-10   4.140672  -4.823536


## 面板运算

需要时间序列和整个截面数据的不同因子间的运算. 面板运算是这里最复杂的一种运算, 同时间序列运算一样也需要指定每个描述子的回溯期数, 系统传递给定义算子的描述子数据是一个二维的面板数据. 面板运算的一个典型例子是进行时间序列和截面的双重标准化.

In [66]:
from QuantStudio.Factor.FactorOperation import PanelOperator
print(qs_help(PanelOperator.calculate))

类型: function
模块: QuantStudio.Factor.FactorOperation
签名: PanelOperator.calculate(self, f: QuantStudio.Factor.Factor.Factor, idt: Union[datetime.datetime, List[datetime.datetime]], iid: Union[str, List[str]], x: list, args: dict)
说明文档:
    算子的运算逻辑实现
    
    Args:
        f: 该算子所属的因子对象
        idt: 当前待计算的时点, 如果 DTMode 为多时点, 则该值为时点序列 list[datetime]
        iid: 当前待计算的 ID, 如果 OutputMode 为全截面, 则该值为 ID 序列 list[str], 该序列在并发时也是全体截面 ID
        x: 描述子当期的数据, [array]
            * 如果 DTMode 为单时点, 那么 x 的第 i 个元素为 array(shape=(LookBack[i]+1, len(iid))), 如果输出形式为全截面返回 array(shape=(len(iid), )), 否则返回单个值
            * 如果 DTMode 为多时点, 那么 x 的第 i 个元素为 array(shape=(LookBack[i]+len(idt), len(iid))), 如果输出形式为全截面返回 array(shape=(len(idt), len(iid))), 否则返回 array(shape=(len(idt), ))
        args: 计算需要附加的模型参数, 来自于算子和因子对象的 ModelArgs, {参数名: 参数值}
    
    Returns:
        在时点 idt, ID 为 iid 的因子值


面板算子的主要参数是 DTMode, LookBack, StartDT, iInitFactor, 其不同的取值会影响传入计算函数的入参，这些参数跟时序算子的作用类似。不再赘述

In [73]:
# 面板运算
from QuantStudio.Factor.FactorOperation import FactorOperatorized

FT = FDB.getTable(table_name="stock_cn_day_bar")
Close = FT.getFactor("close")

@FactorOperatorized(operator_type="Panel", args={"Name": "PanelFun", "Arity": 1, "DataType": "double", "DTMode": "单时点", "LookBack": [3-1], "ModelArgs": {}})
def PanelFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")
    for i, ix in enumerate(x):
        print(f"x[{i}] : {ix}")
    print(f"args : {args}")
    print("-" * 10)
    Tmp = (x[0][-1] - np.nanmean(x[0], axis=0)) / np.nanstd(x[0], axis=0)
    return (Tmp - np.nanmean(Tmp)) / np.nanstd(Tmp)

PanelFactor = PanelFun(Close)

SectionIDs = ["000001.SZ", "000002.SZ", "000003.SZ"]
IDs = ["000001.SZ", "000002.SZ"]
DTRuler = FT.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 10))
DTs = DTRuler[-3:]
print(PanelFactor.readData(ids=IDs, dts=DTs, section_ids=SectionIDs, dt_ruler=DTRuler))

idt : [datetime.datetime(2025, 1, 6, 0, 0), datetime.datetime(2025, 1, 7, 0, 0), datetime.datetime(2025, 1, 8, 0, 0)]
iid : ['000001.SZ', '000002.SZ', '000003.SZ']
x[0] : [[2.30400314 4.26691412 6.10489735]
 [7.08360267 3.92758946 0.2927094 ]
 [8.16825141 5.39536605 0.45850355]]
args : {}
----------
idt : [datetime.datetime(2025, 1, 7, 0, 0), datetime.datetime(2025, 1, 8, 0, 0), datetime.datetime(2025, 1, 9, 0, 0)]
iid : ['000001.SZ', '000002.SZ', '000003.SZ']
x[0] : [[7.08360267 3.92758946 0.2927094 ]
 [8.16825141 5.39536605 0.45850355]
 [3.18402935 7.28947696 5.69195972]]
args : {}
----------
idt : [datetime.datetime(2025, 1, 8, 0, 0), datetime.datetime(2025, 1, 9, 0, 0), datetime.datetime(2025, 1, 10, 0, 0)]
iid : ['000001.SZ', '000002.SZ', '000003.SZ']
x[0] : [[8.16825141 5.39536605 0.45850355]
 [3.18402935 7.28947696 5.69195972]
 [9.15652841 0.19232059 5.69872151]]
args : {}
----------
            000001.SZ  000002.SZ
2025-01-08   0.422776   0.957348
2025-01-09  -1.412806   0.6517

# 运算符重载

对于因子对象, QuantStudio 在 `QuantStudio.Factor.BasicOperator` 中实现了大多数运算符的重载。

重载的运算符如下所示(A, B, C, …表示因子对象, 或者具体的标量数据), 使用运算符构建表达式的输出结果本质上是基于单点运算构建的衍生因子:
* A + B: 将因子 A 数据和因子 B 数据对应相加, 返回新因子
* A - B: 将因子 A 数据和因子 B 数据对应相减, 返回新因子
* A * B: 将因子 A 数据和因子 B 数据对应相乘, 返回新因子
* A / B: 将因子 A 数据和因子 B 数据对应相除, 返回新因子
* A // B: 将因子 A 数据和因子 B 数据对应做向下取整除法, 返回新因子
* A % B: 将因子 A 数据和因子 B 数据对应取余, 返回新因子
* A ** B: 将因子 A 数据和因子 B 数据对应取乘方, A 是底数, B 是幂次, 返回新因子
* A < B: 将因子 A 数据和因子 B 数据对应取小于运算, 结果是 True 或者 False, 返回新因子
* A <= B: 将因子 A 数据和因子 B 数据对应取小于等于运算, 结果是 True 或者 False, 返回新因子
* A > B: 将因子 A 数据和因子 B 数据对应取大于运算, 结果是 True 或者 False, 返回新因子
* A >= B: 将因子 A 数据和因子 B 数据对应取大于等于运算, 结果是 True 或者 False, 返回新因子
* A == B: 将因子 A 数据和因子 B 数据对应是否相等, 结果是 True 或者 False, 返回新因子
* A != B: 将因子 A 数据和因子 B 数据对应是否不等, 结果是 True 或者 False, 返回新因子
* A & B: 将因子 A 数据和因子 B 数据对应做并操作, A 和 B 的数据必须为 True 或者 False的逻辑值, 返回新因子
* A | B: 将因子 A 数据和因子 B 数据对应做或操作, A 和 B 的数据必须为 True 或者 False的逻辑值, 返回新因子
* A ^ B: 将因子 A 数据和因子 B 数据对应做异或操作, A 和 B 的数据必须为 True 或者 False的逻辑值, 返回新因子
* ~A: 将因子 A 数据做取反操作, A 的数据必须为 True 或者 False的逻辑值, 返回新因子
* abs(A): 将因子 A 数据取绝对值, 返回新因子
* -A: 将因子 A 数据取相反数, 返回新因子

另外，`QuantStudio.Factor.BasicOperator` 中还有一个常用的算子 `rename` 主要用于给因子对象重命名。

In [45]:
# 运算符重载
from QuantStudio.Factor.BasicOperator import rename

FT = FDB.getTable(table_name="stock_cn_day_bar")
High, Low = FT.getFactor("high"), FT.getFactor("low")

Mid = rename((High + Low) / 2, factor_name="Mid")
print(Mid.Name)

IDs = ["000001.SZ", "000002.SZ"]
DTs = FT.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 3))
print(Mid.readData(ids=IDs, dts=DTs))

Mid
            000001.SZ  000002.SZ
2025-01-01   6.801660   4.476960
2025-01-02   5.069191   4.899096
2025-01-03   4.807258   2.502005


# 内置算子

内置算子定义在模块 `QuantStudio.Factor.FactorOperator` 中, 预定义了一些比较常用的算子。

In [ ]:
# 使用内置算子: 以 Log 为例
from QuantStudio.Factor.FactorOperator import Log

log = Log(base=np.e)
print(qs_help(log))

FT = FDB.getTable(table_name="stock_cn_day_bar")
Close = FT.getFactor("close")

F = log(Close)

IDs = ["000001.SZ", "000002.SZ"]
DTs = FT.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 3))
print("-" * 10)
print(F.readData(ids=IDs, dts=DTs))

类型: Log
模块: QuantStudio.Factor.FactorOperator
QS 对象类型: 因子算子
QS 对象名称: log
QSID: 4f109a4810623120fe6849b9960d348ef152daf8498591e770762971c071d8be
参数集:
    * OperatorType(算子类型): typing.Literal['Point'], 默认值 'Point', 当前取值: 'Point'
    * Name(名称): <class 'str'>, 默认值 'PointOperator', 当前取值: 'log'
    * ModelArgs(模型参数): typing.Dict[str, typing.Any], 默认值 {}, 当前取值: 
        * base: 2.718281828459045
    * Arity(入参数): typing.Optional[int], 默认值 None, 当前取值: 1
    * DataType(数据类型): typing.Literal['double', 'string', 'object'], 默认值 'double', 当前取值: 'double'
    * Description(描述信息): <class 'str'>, 默认值 '', 当前取值: ''
    * Meta(元信息): typing.Dict[str, typing.Any], 默认值 {}, 当前取值: {}
    * InputFormat(输入格式): typing.Literal['numpy', 'pandas'], 默认值 'numpy', 当前取值: 'numpy'
    * ExpandDescriptors(展开描述子): typing.List[int], 默认值 [], 当前取值: []
    * DescriptorCompoundType(描述子复合类型): typing.List[typing.List[typing.Tuple[str, typing.Literal['double', 'string', 'object']]]], 默认值 [], 当前取值: []
    * MultiMapping(多重映射): <cla

<a id='DataFactor'></a>

# 数据因子

In [6]:
from QuantStudio.Factor.Factor import DataFactor

F = DataFactor(data=1)
print(qs_help(F))

类型: DataFactor
模块: QuantStudio.Factor.Factor
QS 对象类型: 计算节点-因子
QS 对象名称: 1
QSID: de28cdecc0d4c0007d8c7f304678e81d96a3f18a85f3f679c46e3a0b2548a02e
参数集:
    * Name(名称): <class 'str'>, 默认值 'DataFactor', 当前取值: '1'
    * Meta(元信息): <class 'dict'>, 默认值 {}, 当前取值: {}
    * SectionIDs(截面ID): typing.Optional[typing.List[str]], 默认值 None, 当前取值: None
    * CalcDTRuler(计算时点标尺): typing.Optional[typing.List[datetime.datetime]], 默认值 None, 当前取值: None
    * CacheEnabled(启用缓存): <class 'bool'>, 默认值 True, 当前取值: True
    * DataType(数据类型): typing.Literal['double', 'string', 'object'], 默认值 'double', 当前取值: 'double'
    * LookBack(回溯天数): <class 'int'>, 默认值 0, 当前取值: 0
说明文档:
    数据因子: 直接赋予数据产生的因子


In [74]:
IDs = ["000001.SZ", "600519.SH"]
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(3)]

F = DataFactor(data=1)
print(F.readData(ids=IDs, dts=DTs))

print("-" * 10)
F = DataFactor(data=pd.DataFrame(np.random.randn(2, 2), index=DTs[:2], columns=IDs))
print(F.readData(ids=IDs, dts=DTs))

            000001.SZ  600519.SH
2025-01-01        1.0        1.0
2025-01-02        1.0        1.0
2025-01-03        1.0        1.0
----------
            000001.SZ  600519.SH
2025-01-01   0.154491  -0.974294
2025-01-02   1.830922  -1.053963
2025-01-03        NaN        NaN
